In [1]:
import pandas as pd
df = pd.read_csv('LIAR_bal.csv')
from sklearn.model_selection import train_test_split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['title'], 
    df['label'],            
    test_size=0.2,               
    random_state=42,               
)

In [2]:
print("Train Samples:", len(X_train_text))
print("Test Samples :", len(X_test_text))

Train Samples: 8868
Test Samples : 2218


In [3]:
# =========================
# BERT-Base + Hyperparameter Tuning Pipeline
# =========================

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

# ----------------------------------------------------------
# LOAD BERT
# ----------------------------------------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bert_model.to(device)
bert_model.eval()

print("Device:", device)

# ----------------------------------------------------------
# DATASET CLASS
# ----------------------------------------------------------
class TextDataset(Dataset):

    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

# ----------------------------------------------------------
# FUNCTION TO EXTRACT BERT EMBEDDINGS
# ----------------------------------------------------------
def get_bert_embeddings(texts, batch_size=16):

    dataset = TextDataset(texts, tokenizer)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    embeddings = []

    with torch.no_grad():

        for batch in tqdm(loader, desc="Generating BERT Embeddings"):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = bert_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            # CLS token embedding
            cls_embedding = outputs.last_hidden_state[:, 0, :]

            embeddings.append(cls_embedding.cpu().numpy())

    return np.vstack(embeddings)

# ----------------------------------------------------------
# EXTRACT TRAIN / TEST EMBEDDINGS
# ----------------------------------------------------------
print("\nExtracting Train Embeddings...")
X_train = get_bert_embeddings(X_train_text)

print("\nExtracting Test Embeddings...")
X_test = get_bert_embeddings(X_test_text)

print("\nTrain Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

# ==========================================================
# MODELS + PARAMETER GRID
# ==========================================================
param_grid = {

    "Logistic Regression": {
        "model": LogisticRegression(
            max_iter=3000,
            random_state=42
        ),
        "params": {
            "C": [0.1, 0.5, 1, 2, 5]
        }
    },

    "SVM Linear": {
        "model": SVC(
            kernel="linear",
            random_state=42
        ),
        "params": {
            "C": [0.1, 0.5, 1, 2, 5]
        }
    },

    "SVM RBF": {
        "model": SVC(
            kernel="rbf",
            random_state=42
        ),
        "params": {
            "C": [0.5, 1, 2, 5],
            "gamma": ["scale", "auto", 0.1]
        }
    },

    "Random Forest": {
        "model": RandomForestClassifier(
            random_state=42
        ),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [None, 10, 20]
        }
    }
}

# ==========================================================
# STEP 1: GRID SEARCH WITH 5-FOLD CV
# ==========================================================
print("\n" + "="*60)
print("MODEL SELECTION WITH PROPER 5-FOLD CV")
print("="*60)

results = []
best_models = {}

for model_name, mp in param_grid.items():

    print(f"\nTuning: {model_name}")

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    grid = GridSearchCV(
        estimator=mp["model"],
        param_grid=mp["params"],
        cv=skf,
        scoring="accuracy",
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)

    best_models[model_name] = grid.best_estimator_

    results.append({
        "Model": model_name,
        "Best Params": grid.best_params_,
        "Train CV Accuracy": round(grid.best_score_, 4)
    })

    print("Best CV Accuracy:", round(grid.best_score_, 4))
    print("Best Params     :", grid.best_params_)

# ==========================================================
# STEP 2: RANK MODELS
# ==========================================================
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(
    by="Train CV Accuracy",
    ascending=False
)

print("\n" + "="*60)
print("MODEL RANKING")
print("="*60)

print(df_results)

best_model_name = df_results.iloc[0]["Model"]
best_model = best_models[best_model_name]

print("\nSelected Best Model:", best_model_name)

# ==========================================================
# STEP 3: RETRAIN BEST MODEL ON FULL TRAIN DATA
# ==========================================================
print("\n" + "="*60)
print("RETRAINING BEST MODEL")
print("="*60)

best_model.fit(X_train, y_train)

print("Model Retrained Successfully")

# ==========================================================
# STEP 4: FINAL TEST EVALUATION
# ==========================================================
print("\n" + "="*60)
print("FINAL TEST EVALUATION")
print("="*60)

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average="binary")

print("\nFinal Test Accuracy :", round(test_accuracy, 4))
print("Final Test F1 Score :", round(test_f1, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ==========================================================
# STEP 5: COMPARE ALL MODELS ON TEST SET
# ==========================================================
print("\n" + "="*60)
print("COMPARISON OF ALL MODELS ON TEST SET")
print("="*60)

comparison_results = []

for model_name, estimator in best_models.items():

    estimator.fit(X_train, y_train)

    pred = estimator.predict(X_test)

    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average="binary")

    comparison_results.append({
        "Model": model_name,
        "Test Accuracy": round(acc, 4),
        "Test F1": round(f1, 4)
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df.sort_values(
    by=["Test F1", "Test Accuracy"],
    ascending=False
)

print(comparison_df)

# ==========================================================
# STEP 6: BEST MODEL BASED ON TEST SET
# ==========================================================
best_test_model_name = comparison_df.iloc[0]["Model"]
best_test_acc = comparison_df.iloc[0]["Test Accuracy"]
best_test_f1 = comparison_df.iloc[0]["Test F1"]

print("\n" + "="*60)
print("BEST MODEL BASED ON TEST SET")
print("="*60)

print("Best Model     :", best_test_model_name)
print("Test Accuracy  :", best_test_acc)
print("Test F1 Score  :", best_test_f1)

Device: cuda

Extracting Train Embeddings...


Generating BERT Embeddings:   0%|          | 0/555 [00:00<?, ?it/s]


Extracting Test Embeddings...


Generating BERT Embeddings:   0%|          | 0/139 [00:00<?, ?it/s]


Train Shape: (8868, 768)
Test Shape : (2218, 768)

MODEL SELECTION WITH PROPER 5-FOLD CV

Tuning: Logistic Regression
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best CV Accuracy: 0.6001
Best Params     : {'C': 0.1}

Tuning: SVM Linear
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best CV Accuracy: 0.5947
Best Params     : {'C': 0.1}

Tuning: SVM RBF
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best CV Accuracy: 0.6093
Best Params     : {'C': 2, 'gamma': 'auto'}

Tuning: Random Forest
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best CV Accuracy: 0.6006
Best Params     : {'max_depth': 20, 'n_estimators': 200}

MODEL RANKING
                 Model                             Best Params  \
2              SVM RBF               {'C': 2, 'gamma': 'auto'}   
3        Random Forest  {'max_depth': 20, 'n_estimators': 200}   
0  Logistic Regression                              {'C': 0.1}   
1           SVM Linear                    

In [4]:
import pickle

with open("best_bert_embed_liar.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Best model saved successfully")

Best model saved successfully
